# Vector Stores

Cómo indexar embeddings y buscar a escala. Usamos los **mismos** documentos en dos vectorstores distintos (FAISS y Chroma) y comparamos.

**Nota**: chunking real, persistencia, filtros por metadata y carga de PDFs los veremos en el **módulo de RAG**, que es donde de verdad importan.

## Dependencias

In [1]:
%pip install -q \
  "langchain>=1.0" \
  "langchain-core>=1.0" \
  "langchain-openai>=1.0" \
  "langchain-community>=0.4" \
  "langchain-chroma>=1.0" \
  "openai>=2.0" \
  faiss-cpu chromadb datasets tiktoken

## Setup · API key de OpenAI

In [2]:
import os, getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

## Cargar corpus · Wikipedia (Cohere dataset)

Streaming de los primeros 2000 **fragmentos** (párrafos) de la Wikipedia en inglés simple (`simple`), publicados en Hugging Face. Cada fila es un párrafo de un artículo, así que un mismo artículo (p. ej. *Alan Turing*) aparece en varias filas. El dataset trae también embeddings de Cohere precalculados, pero no los usamos: solo nos interesan los textos (los embebemos con OpenAI).

In [3]:
from datasets import load_dataset
from langchain_core.documents import Document

max_docs = 2000
docs_stream = load_dataset(
    "Cohere/wikipedia-2023-11-embed-multilingual-v3",
    "simple",
    split="train",
    streaming=True,
)

documents = []
for doc in docs_stream:
    documents.append(Document(
        page_content=doc["text"],
        metadata={"title": doc["title"], "url": doc["url"]},
    ))
    if len(documents) >= max_docs:
        break

print(f"cargados {len(documents)} documentos")

cargados 2000 documentos


## Modelo de embedding · OpenAI `text-embedding-3-small`

Especificamos el modelo explícitamente. El default de `OpenAIEmbeddings()` antiguo era `text-embedding-ada-002` (2022); hoy lo correcto es `text-embedding-3-small` (más barato y mejor) o `text-embedding-3-large` (mejor aún, más caro).

In [4]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## Vectorstore 1 · FAISS

Local, in-memory, muy rápido. Ideal para prototipos y datasets que caben en RAM.

In [5]:
from langchain_community.vectorstores import FAISS

faiss_db = FAISS.from_documents(documents, embeddings)
print(f"FAISS index creado · {faiss_db.index.ntotal} vectores")

FAISS index creado · 2000 vectores


## Vectorstore 2 · Chroma

Open source, persistible en SQLite, API más expresiva. Ideal para proyectos con metadata rica o que necesiten persistir y recargar.

In [6]:
from langchain_chroma import Chroma

chroma_db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="wiki_simple_demo",
)
print(f"Chroma collection creada")

Chroma collection creada


## Comparativa · misma query, ambos stores

Comparemos los top-3 resultados que devuelve cada uno para la misma pregunta. Deberían coincidir bastante (los embeddings son los mismos, solo cambia el algoritmo de indexación).

Ojo al leer el número entre corchetes: en ambos stores es una **distancia** (L2 al cuadrado: menor = más parecido), no una similitud.

In [7]:
def show_results(name, hits):
    print(f"\n=== {name} ===")
    for doc, score in hits:
        title = doc.metadata.get("title", "?")
        snippet = doc.page_content[:120].replace("\n", " ")
        print(f"  [{score:.3f}] {title} · {snippet}...")

queries = [
    "Who is Alan Turing?",
    "What is the population of Spain?",
    "Tell me about the Eiffel Tower",
]

for q in queries:
    print(f"\n{'='*70}\nQUERY: {q}\n{'='*70}")
    show_results("FAISS", faiss_db.similarity_search_with_score(q, k=3))
    show_results("Chroma", chroma_db.similarity_search_with_score(q, k=3))


QUERY: Who is Alan Turing?



=== FAISS ===
  [0.507] Alan Turing · Alan Mathison Turing OBE FRS (London, 23 June 1912 – Wilmslow, Cheshire, 7 June 1954) was an English mathematician and c...
  [0.630] Alan Turing · Alan was a brilliant mathematician and cryptographer. He became the founder of modern-day computer science and artificia...
  [0.632] Alan Turing · Jack Copeland 2012. Alan Turing: The codebreaker who saved 'millions of lives'. BBC News / Technology...



=== Chroma ===
  [0.507] Alan Turing · Alan Mathison Turing OBE FRS (London, 23 June 1912 – Wilmslow, Cheshire, 7 June 1954) was an English mathematician and c...
  [0.630] Alan Turing · Alan was a brilliant mathematician and cryptographer. He became the founder of modern-day computer science and artificia...
  [0.632] Alan Turing · Jack Copeland 2012. Alan Turing: The codebreaker who saved 'millions of lives'. BBC News / Technology...

QUERY: What is the population of Spain?



=== FAISS ===
  [1.116] History of Spain · There was not much peace in Spain during the first part of the 20th century. Some Spaniards tried to set up a government...
  [1.118] Autonomous communities of Spain · Spain is divided in 17 parts called autonomous communities. Autonomous means that each of these autonomous communities h...
  [1.127] Autonomous communities of Spain · Spain has fifty smaller parts called provinces. In 1978 these parts came together, making the autonomous communities....



=== Chroma ===
  [1.116] History of Spain · There was not much peace in Spain during the first part of the 20th century. Some Spaniards tried to set up a government...
  [1.117] Autonomous communities of Spain · Spain is divided in 17 parts called autonomous communities. Autonomous means that each of these autonomous communities h...
  [1.127] Autonomous communities of Spain · Spain has fifty smaller parts called provinces. In 1978 these parts came together, making the autonomous communities....

QUERY: Tell me about the Eiffel Tower



=== FAISS ===
  [1.012] France · Some of the most famous attractions in Paris, are the Eiffel Tower and the Arc de Triomphe. Another one is Mont Saint Mi...
  [1.159] Architecture · Towards the end of the 19th Century with a second Industrial Revolution, steel became much cheaper. Architects began to ...
  [1.325] France · A European Disneyland is located in a suburb east of Paris. The resort opened in 1992 and is also a popular tourist dest...



=== Chroma ===
  [1.012] France · Some of the most famous attractions in Paris, are the Eiffel Tower and the Arc de Triomphe. Another one is Mont Saint Mi...
  [1.160] Architecture · Towards the end of the 19th Century with a second Industrial Revolution, steel became much cheaper. Architects began to ...
  [1.325] France · A European Disneyland is located in a suburb east of Paris. The resort opened in 1992 and is also a popular tourist dest...


## ¿Cuál uso?

Para esta demo tan simple, son equivalentes. Las diferencias importan cuando creces:

| Aspecto | FAISS | Chroma |
|---|---|---|
| Persistencia | manual (`save_local`) | de serie |
| Filtros por metadata | limitados | sí, con operadores |
| Cliente/servidor | no | sí (modo cliente) |
| Latencia in-process | excelente | muy buena |
| Curva de entrada | mínima | mínima |

**Persistencia, chunking real y filtros por metadata se ven en el módulo de RAG**, que es donde de verdad importan.

## Cierre

- Un vectorstore es una **DB optimizada** para búsqueda por similitud
- FAISS y Chroma son intercambiables a este nivel · cambias 2 imports
- En el módulo de RAG sumamos las piezas reales de un RAG: PDF loaders, chunking estrategias, retrieval con metadata, reranking